# Notebook 44 - UltraTrack KLT oracle-mask gate

Notebook 43 separated the downstream work into two gates. This notebook starts the first one: an UltraTrack/KLT parity prototype.

Scope:

1. Rebuild the MATLAB Hough-local tracking masks from `UTT_numeric_export.mat`.
2. Use OpenCV equivalents for MATLAB `detectMinEigenFeatures`, `vision.PointTracker`, and affine propagation.
3. Save fascicle and aponeurosis geometry before Kalman filtering.
4. Compare pixel-by-pixel geometry against MATLAB `Region.Fascicle.fas_x_original` / `fas_y_original` and `Region.sup/deep` pre-Kalman aponeurosis tracks.

This is intentionally an **oracle-mask** notebook: it uses MATLAB's exported `geofeatures` to build the same KLT masks. That isolates the KLT/affine step before we move the implementation into `ultrasound_tracker` and replace the oracle masks with Python-generated masks.

In [1]:
from pathlib import Path
import json
import os
import sys
import time

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import cv2
import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row

OUT = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
VIDEO_PATH = ROOT / "data" / "raw" / "Test2.mp4"
NB43_KLT_METRICS = ROOT / "results" / "notebook43_klt_kalman_downstream_gates" / "klt_gate_metrics.csv"

for path in [MATLAB_EXPORT, VIDEO_PATH, NB43_KLT_METRICS]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Output folder:", OUT)
print("MATLAB export:", MATLAB_EXPORT)
print("Video:", VIDEO_PATH)

Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate
MATLAB export: /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
Video: /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4


## Load MATLAB KLT contract variables

MATLAB stores fascicle endpoints as `[deep; superficial]`. The Python comparison convention is `[x_sup, y_sup, x_deep, y_deep]`, so the loader below flips the endpoint order.

For Hough-local UltraTrack, the relevant MATLAB code path is:

- Build `I_fmasked` from narrow bands around `geofeatures(f).x/y` peak lines.
- Intersect it with the corrected Hough fascicle ROI using `super_pos`, `deep_pos`, and `r = 0.1`.
- Build `I_amasked` from the superficial/deep aponeurosis cut bands.
- Detect features with `detectMinEigenFeatures(..., FilterSize=11, MinQuality=0.005)`.
- Track with `vision.PointTracker(NumPyramidLevels=4, MaxIterations=50, MaxBidirectionalError=inf, BlockSize=[21 71])`.
- Fit a full affine transform using `estimateGeometricTransform2D(..., 'affine', 'MaxDistance', 50)`.

In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
geofeatures = list(np.asarray(mat_root["geofeatures"], dtype=object).reshape(-1))
parms = mat_root["parms"]

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height
block_size_matlab = np.asarray(mat_root["BlockSize"], dtype=int).reshape(-1)
contract_win_size = (int(block_size_matlab[1]), int(block_size_matlab[0]))

cap = cv2.VideoCapture(str(VIDEO_PATH))
video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = float(cap.get(cv2.CAP_PROP_FPS))
cap.release()

if (video_height, video_width) != (height, width):
    raise ValueError(f"Video shape {(video_height, video_width)} != MATLAB shape {(height, width)}")

print({
    "matlab_frames": mat_n,
    "video_frames": video_frames,
    "shape": (height, width),
    "fps": video_fps,
    "mm_per_px": mm_per_px,
    "matlab_block_size_height_width": block_size_matlab.tolist(),
    "opencv_contract_win_size_width_height": contract_win_size,
})

{'matlab_frames': 2666, 'video_frames': 2667, 'shape': (562, 706), 'fps': 33.341, 'mm_per_px': 0.09021352313167261, 'matlab_block_size_height_width': [21, 71], 'opencv_contract_win_size_width_height': (71, 21)}


In [3]:
def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        if arr.size < 2:
            out.append([np.nan, np.nan])
        else:
            out.append(arr[:2])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_apo_segments(prefix: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(region[f"{prefix}_x"], n=n)
    y = matlab_pair_series(region[f"{prefix}_y"], n=n)
    return np.column_stack([x[:, 0], y[:, 0], x[:, 1], y[:, 1]])


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


mat_klt_segments = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_sup_segments = matlab_apo_segments("sup", n=mat_n)
mat_deep_segments = matlab_apo_segments("deep", n=mat_n)

print("MATLAB pre-Kalman fascicle segment[0]:", mat_klt_segments[0])
print("MATLAB superficial apo segment[0]:", mat_sup_segments[0])
print("MATLAB deep apo segment[0]:", mat_deep_segments[0])

MATLAB pre-Kalman fascicle segment[0]: [733.          54.41875512 -27.         309.01343161]
MATLAB superficial apo segment[0]: [  1.          36.85315315 706.          53.77084357]
MATLAB deep apo segment[0]: [  1.         309.92612613 706.         332.90647011]


## Rebuild MATLAB tracking masks

The masks here are for KLT feature detection only. They are separate from the TimTrack mask parity notebooks because UltraTrack uses MATLAB `geofeatures` to build tracking masks after TimTrack has already produced local Hough lines and aponeurosis positions.

In [4]:
super_cut = np.asarray(parms["apo"]["super"]["cut"], dtype=float).reshape(-1)
deep_cut = np.asarray(parms["apo"]["deep"]["cut"], dtype=float).reshape(-1)


def poly_mask_1b(x_values, y_values, shape=(height, width)) -> np.ndarray:
    """Approximate MATLAB poly2mask for 1-based x/y polygon coordinates."""
    x = np.asarray(x_values, dtype=float).reshape(-1) - 1.0
    y = np.asarray(y_values, dtype=float).reshape(-1) - 1.0
    points = np.column_stack([x, y])
    points = np.rint(points).astype(np.int32)
    mask = np.zeros(shape, dtype=np.uint8)
    cv2.fillPoly(mask, [points], 1)
    return mask.astype(bool)


def tracking_masks(frame_index: int) -> dict[str, np.ndarray]:
    entry = geofeatures[frame_index]

    line_mask = np.zeros((height, width), dtype=bool)
    xs = np.asarray(entry["x"], dtype=float)
    ys = np.asarray(entry["y"], dtype=float)
    for (x1, x2), (y1, y2) in zip(xs, ys):
        px = [x1, x1, x2, x2]
        py = [y1 - 5.0, y1 + 5.0, y2 + 5.0, y2 - 5.0]
        if np.all(np.isfinite(px)) and np.all(np.isfinite(py)):
            line_mask |= poly_mask_1b(px, py)

    super_pos = np.asarray(entry["super_pos"], dtype=float).reshape(-1)
    deep_pos = np.asarray(entry["deep_pos"], dtype=float).reshape(-1)
    thickness = deep_pos - super_pos
    r = 0.1

    roix = [1.0, 1.0, float(width), float(width), 1.0]
    roiy_fcor = np.rint([
        super_pos[0] + thickness[0] * r,
        deep_pos[0] - thickness[0] * r,
        deep_pos[1] - thickness[1] * r,
        super_pos[1] + thickness[1] * r,
        super_pos[0] + thickness[0] * r,
    ])
    fcor_mask = poly_mask_1b(roix, roiy_fcor)
    fascicle_mask = line_mask & fcor_mask

    super_y = [super_cut[0] * height, super_cut[1] * height, super_cut[1] * height, super_cut[0] * height, super_cut[0] * height]
    deep_y = [deep_cut[0] * height, deep_cut[1] * height, deep_cut[1] * height, deep_cut[0] * height, deep_cut[0] * height]
    super_mask = poly_mask_1b(roix, super_y)
    deep_mask = poly_mask_1b(roix, deep_y)
    apo_mask = super_mask | deep_mask

    return {
        "line_mask": line_mask,
        "fcor_mask": fcor_mask,
        "fascicle_mask": fascicle_mask,
        "super_mask": super_mask,
        "deep_mask": deep_mask,
        "apo_mask": apo_mask,
    }


def masked_image(gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    out = np.asarray(gray).copy()
    out[~mask] = 0
    return out


mask_audit_frames = [0, 122, 533, 691, 955, 1066, 1600, 2133, 2665]
mask_rows = []
for frame in mask_audit_frames:
    m = tracking_masks(frame)
    mask_rows.append({
        "frame": frame,
        "line_pixels": int(m["line_mask"].sum()),
        "fcor_pixels": int(m["fcor_mask"].sum()),
        "fascicle_pixels": int(m["fascicle_mask"].sum()),
        "apo_pixels": int(m["apo_mask"].sum()),
        "super_apo_pixels": int(m["super_mask"].sum()),
        "deep_apo_pixels": int(m["deep_mask"].sum()),
    })
mask_audit = pd.DataFrame(mask_rows)
mask_audit_path = OUT / "tracking_mask_pixel_counts.csv"
mask_audit.to_csv(mask_audit_path, index=False)
print("Saved:", mask_audit_path)
display(mask_audit)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/tracking_mask_pixel_counts.csv


,frame,line_pixels,fcor_pixels,fascicle_pixels,apo_pixels,super_apo_pixels,deep_apo_pixels
0,0,44254,156732,43468,106606,50126,56480
1,122,30015,149672,27431,106606,50126,56480
2,533,40286,156379,39029,106606,50126,56480
3,691,26771,147201,23695,106606,50126,56480
4,955,28637,150378,25916,106606,50126,56480
5,1066,41751,156732,40656,106606,50126,56480
6,1600,44048,156379,42901,106606,50126,56480
7,2133,40077,156379,38770,106606,50126,56480
8,2665,42815,157085,41551,106606,50126,56480


## OpenCV approximation of MATLAB KLT

OpenCV does not expose MathWorks `vision.PointTracker` or `detectMinEigenFeatures` exactly, so this is the closest prototype:

- `cv2.goodFeaturesToTrack(..., qualityLevel=0.005, blockSize=11, minDistance=1, useHarrisDetector=False)` for MinEigen/Shi-Tomasi corners.
- `maxCorners=300` for fascicle points to mirror `selectStrongest(300)`.
- `maxCorners=0` for aponeurosis points so the detector can return all qualified points.
- `cv2.calcOpticalFlowPyrLK(..., maxLevel=3, criteria=(50, 0.01))` for 4 pyramid levels including level 0.
- `cv2.estimateAffine2D(..., ransacReprojThreshold=50)` for full affine propagation.

The affine fit is done in MATLAB-style 1-based coordinates by adding 1 to OpenCV point locations before estimating the transform.

In [5]:
def read_gray_frames(limit: int | None = None) -> list[np.ndarray]:
    n = mat_n if limit is None else min(int(limit), mat_n)
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    frames = []
    for _ in range(n):
        ok, frame = cap.read()
        if not ok:
            break
        if frame.ndim == 3:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
        else:
            frames.append(frame.copy())
    cap.release()
    if len(frames) != n:
        raise RuntimeError(f"Read {len(frames)} frames, expected {n}")
    return frames


def detect_min_eigen_like(gray: np.ndarray, mask: np.ndarray, max_corners: int) -> np.ndarray:
    img = masked_image(gray, mask)
    points = cv2.goodFeaturesToTrack(
        img,
        maxCorners=int(max_corners),
        qualityLevel=0.005,
        minDistance=1,
        blockSize=11,
        useHarrisDetector=False,
    )
    if points is None:
        return np.empty((0, 1, 2), dtype=np.float32)
    return points.astype(np.float32)


def filter_points_by_mask(points: np.ndarray, mask: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    if len(points) == 0:
        return points.reshape(0, 1, 2)
    xy = np.rint(points).astype(int)
    xy[:, 0] = np.clip(xy[:, 0], 0, width - 1)
    xy[:, 1] = np.clip(xy[:, 1], 0, height - 1)
    keep = mask[xy[:, 1], xy[:, 0]]
    return points[keep].reshape(-1, 1, 2).astype(np.float32)


def track_points(prev_gray: np.ndarray, gray: np.ndarray, points: np.ndarray, win_size: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    points = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)
    if len(points) == 0:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
    new_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        gray,
        points,
        None,
        winSize=win_size,
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 50, 0.01),
    )
    if new_points is None or status is None:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
    keep = status.reshape(-1).astype(bool)
    return points.reshape(-1, 2)[keep], new_points.reshape(-1, 2)[keep]


def estimate_affine_matlab_coords(old_points_0b: np.ndarray, new_points_0b: np.ndarray) -> tuple[np.ndarray | None, int]:
    old_points_0b = np.asarray(old_points_0b, dtype=np.float32).reshape(-1, 2)
    new_points_0b = np.asarray(new_points_0b, dtype=np.float32).reshape(-1, 2)
    if len(old_points_0b) < 3 or len(new_points_0b) < 3:
        return None, 0
    affine, inliers = cv2.estimateAffine2D(
        old_points_0b + 1.0,
        new_points_0b + 1.0,
        method=cv2.RANSAC,
        ransacReprojThreshold=50.0,
        maxIters=2000,
        confidence=0.99,
        refineIters=10,
    )
    if affine is None:
        return None, 0
    n_inliers = int(np.asarray(inliers).sum()) if inliers is not None else len(old_points_0b)
    return affine.astype(np.float32), n_inliers


def apply_affine_1b(points_1b: np.ndarray, affine: np.ndarray) -> np.ndarray:
    points = np.asarray(points_1b, dtype=np.float32).reshape(-1, 2)
    return cv2.transform(points[None, :, :], affine.astype(np.float32))[0].astype(float)


def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or row["rmse"] <= target_rmse)
    return row


def save_metrics(rows: list[dict], path: Path) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    order = ["comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]
    df = df[[col for col in order if col in df.columns]]
    df.to_csv(path, index=False)
    return df

In [6]:
def run_oracle_mask_klt(limit: int | None = None, win_size: tuple[int, int] = contract_win_size, label: str = "contract") -> dict:
    frames = read_gray_frames(limit=limit)
    n = len(frames)
    masks0 = tracking_masks(0)

    fascicle_points = detect_min_eigen_like(frames[0], masks0["fascicle_mask"], max_corners=300)
    apo_points = detect_min_eigen_like(frames[0], masks0["apo_mask"], max_corners=0)
    fascicle_points = filter_points_by_mask(fascicle_points, masks0["fcor_mask"])
    apo_points = filter_points_by_mask(apo_points, masks0["apo_mask"])

    fascicle_segment = mat_klt_segments[0].astype(float).copy()
    sup_segment = mat_sup_segments[0].astype(float).copy()
    deep_segment = mat_deep_segments[0].astype(float).copy()

    out_fascicle = np.full((n, 4), np.nan, dtype=np.float32)
    out_sup = np.full((n, 4), np.nan, dtype=np.float32)
    out_deep = np.full((n, 4), np.nan, dtype=np.float32)
    out_fascicle[0] = fascicle_segment
    out_sup[0] = sup_segment
    out_deep[0] = deep_segment

    f_points_count = np.zeros(n, dtype=np.int32)
    a_points_count = np.zeros(n, dtype=np.int32)
    f_inlier_count = np.zeros(n, dtype=np.int32)
    a_inlier_count = np.zeros(n, dtype=np.int32)
    f_affine_ok = np.zeros(n, dtype=bool)
    a_affine_ok = np.zeros(n, dtype=bool)
    f_reinitialized = np.zeros(n, dtype=bool)
    a_reinitialized = np.zeros(n, dtype=bool)
    f_points_count[0] = len(fascicle_points)
    a_points_count[0] = len(apo_points)

    start = time.time()
    for i in range(1, n):
        prev_gray = frames[i - 1]
        gray = frames[i]

        old_f, new_f = track_points(prev_gray, gray, fascicle_points, win_size=win_size)
        old_a, new_a = track_points(prev_gray, gray, apo_points, win_size=win_size)

        f_affine, f_inliers = estimate_affine_matlab_coords(old_f, new_f)
        a_affine, a_inliers = estimate_affine_matlab_coords(old_a, new_a)

        if f_affine is not None:
            fascicle_points_1b = fascicle_segment.reshape(2, 2)
            fascicle_segment = apply_affine_1b(fascicle_points_1b, f_affine).reshape(4)
            f_affine_ok[i] = True
            f_inlier_count[i] = f_inliers

        if a_affine is not None:
            sup_points_1b = sup_segment.reshape(2, 2)
            deep_points_1b = deep_segment.reshape(2, 2)
            sup_new = apply_affine_1b(sup_points_1b, a_affine)
            deep_new = apply_affine_1b(deep_points_1b, a_affine)
            # MATLAB keeps apo x endpoints fixed at [1, width] after applying the warp.
            sup_segment = np.asarray([1.0, sup_new[0, 1], float(width), sup_new[1, 1]], dtype=float)
            deep_segment = np.asarray([1.0, deep_new[0, 1], float(width), deep_new[1, 1]], dtype=float)
            a_affine_ok[i] = True
            a_inlier_count[i] = a_inliers

        out_fascicle[i] = fascicle_segment
        out_sup[i] = sup_segment
        out_deep[i] = deep_segment

        masks_i = tracking_masks(i)
        fascicle_points = np.asarray(new_f, dtype=np.float32).reshape(-1, 1, 2)
        apo_points = np.asarray(new_a, dtype=np.float32).reshape(-1, 1, 2)

        if len(fascicle_points) < 100:
            fascicle_points = detect_min_eigen_like(gray, masks_i["fascicle_mask"], max_corners=300)
            f_reinitialized[i] = True
        fascicle_points = filter_points_by_mask(fascicle_points, masks_i["fcor_mask"])

        if len(apo_points) < 500:
            apo_points = detect_min_eigen_like(gray, masks_i["apo_mask"], max_corners=0)
            a_reinitialized[i] = True
        apo_points = filter_points_by_mask(apo_points, masks_i["apo_mask"])

        f_points_count[i] = len(fascicle_points)
        a_points_count[i] = len(apo_points)

        if i % 500 == 0 or i == n - 1:
            print(
                f"{label}: processed {i + 1}/{n} frames | "
                f"fpts={f_points_count[i]} apts={a_points_count[i]} "
                f"elapsed={time.time() - start:.1f}s"
            )

    return {
        "label": label,
        "win_size": win_size,
        "frame": np.arange(n, dtype=np.int32),
        "fascicle_segments": out_fascicle,
        "sup_apo_lines": out_sup,
        "deep_apo_lines": out_deep,
        "f_points_count": f_points_count,
        "a_points_count": a_points_count,
        "f_inlier_count": f_inlier_count,
        "a_inlier_count": a_inlier_count,
        "f_affine_ok": f_affine_ok,
        "a_affine_ok": a_affine_ok,
        "f_reinitialized": f_reinitialized,
        "a_reinitialized": a_reinitialized,
        "elapsed_s": time.time() - start,
    }

## Window-size smoke test on the first 300 frames

This is a small parameter sanity check. The full run below uses the MATLAB block-size contract converted to OpenCV's `(width, height)` convention: `[21, 71]` in MATLAB becomes `(71, 21)` for OpenCV.

In [7]:
def klt_metric_rows(result: dict, n: int | None = None, prefix: str = "klt") -> list[dict]:
    est = np.asarray(result["fascicle_segments"], dtype=float)
    if n is not None:
        est = est[:n]
        ref_fas = mat_klt_segments[:n]
        ref_sup = mat_sup_segments[:n]
        ref_deep = mat_deep_segments[:n]
    else:
        ref_fas = mat_klt_segments[: len(est)]
        ref_sup = mat_sup_segments[: len(est)]
        ref_deep = mat_deep_segments[: len(est)]

    sup_est = np.asarray(result["sup_apo_lines"], dtype=float)[: len(est)]
    deep_est = np.asarray(result["deep_apo_lines"], dtype=float)[: len(est)]

    length_target_px = 2.0 / mm_per_px
    return [
        metric(f"{prefix}_x_sup_px", ref_fas[:, 0], est[:, 0], "px", target_rmse=2.0),
        metric(f"{prefix}_y_sup_px", ref_fas[:, 1], est[:, 1], "px", target_rmse=2.0),
        metric(f"{prefix}_x_deep_px", ref_fas[:, 2], est[:, 2], "px", target_rmse=2.0),
        metric(f"{prefix}_y_deep_px", ref_fas[:, 3], est[:, 3], "px", target_rmse=2.0),
        metric(f"{prefix}_angle_deg", normalized_angles_deg(ref_fas), normalized_angles_deg(est), "deg", target_rmse=1.0),
        metric(f"{prefix}_length_px", line_lengths_batch(ref_fas), line_lengths_batch(est), "px", target_rmse=length_target_px),
        metric(f"{prefix}_length_mm", line_lengths_batch(ref_fas) * mm_per_px, line_lengths_batch(est) * mm_per_px, "mm", target_rmse=2.0),
        metric(f"{prefix}_super_y1_px", ref_sup[:, 1], sup_est[:, 1], "px", target_rmse=2.0),
        metric(f"{prefix}_super_y2_px", ref_sup[:, 3], sup_est[:, 3], "px", target_rmse=2.0),
        metric(f"{prefix}_deep_y1_px", ref_deep[:, 1], deep_est[:, 1], "px", target_rmse=2.0),
        metric(f"{prefix}_deep_y2_px", ref_deep[:, 3], deep_est[:, 3], "px", target_rmse=2.0),
    ]


sweep_windows = [contract_win_size, (21, 71), (21, 21), (31, 31)]
sweep_rows = []
for win in sweep_windows:
    result = run_oracle_mask_klt(limit=300, win_size=win, label=f"sweep_{win[0]}x{win[1]}")
    rows = klt_metric_rows(result, prefix="sweep")
    by_name = {row["comparison"]: row for row in rows}
    sweep_rows.append({
        "win_width": win[0],
        "win_height": win[1],
        "is_contract_win_size": win == contract_win_size,
        "n": len(result["frame"]),
        "f_affine_rate": float(np.mean(result["f_affine_ok"][1:])),
        "a_affine_rate": float(np.mean(result["a_affine_ok"][1:])),
        "mean_f_points": float(np.mean(result["f_points_count"])),
        "mean_a_points": float(np.mean(result["a_points_count"])),
        "f_reinit_count": int(np.sum(result["f_reinitialized"])),
        "a_reinit_count": int(np.sum(result["a_reinitialized"])),
        "angle_rmse_deg": by_name["sweep_angle_deg"]["rmse"],
        "length_rmse_px": by_name["sweep_length_px"]["rmse"],
        "length_rmse_mm": by_name["sweep_length_mm"]["rmse"],
        "super_y_rmse_px": max(by_name["sweep_super_y1_px"]["rmse"], by_name["sweep_super_y2_px"]["rmse"]),
        "deep_y_rmse_px": max(by_name["sweep_deep_y1_px"]["rmse"], by_name["sweep_deep_y2_px"]["rmse"]),
    })

sweep_df = pd.DataFrame(sweep_rows)
sweep_path = OUT / "klt_oracle_mask_window_sweep_300.csv"
sweep_df.to_csv(sweep_path, index=False)
print("Saved:", sweep_path)
display(sweep_df)

sweep_71x21: processed 300/300 frames | fpts=264 apts=724 elapsed=1.4s


sweep_21x71: processed 300/300 frames | fpts=254 apts=692 elapsed=1.3s


sweep_21x21: processed 300/300 frames | fpts=258 apts=623 elapsed=1.0s


sweep_31x31: processed 300/300 frames | fpts=255 apts=678 elapsed=1.2s
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/klt_oracle_mask_window_sweep_300.csv


,win_width,win_height,is_contract_win_size,n,f_affine_rate,a_affine_rate,mean_f_points,mean_a_points,f_reinit_count,a_reinit_count,angle_rmse_deg,length_rmse_px,length_rmse_mm,super_y_rmse_px,deep_y_rmse_px
0,71,21,True,300,1.0,1.0,272.520000,772.780000,0,0,4.169620,111.502411,10.059025,2.621669,12.034452
1,21,71,False,300,1.0,1.0,267.730000,752.546667,0,0,4.156302,114.590834,10.337643,1.983615,11.804160
2,21,21,False,300,1.0,1.0,269.143333,710.440000,0,0,4.148915,113.256406,10.217259,2.067102,11.406031
3,31,31,False,300,1.0,1.0,267.546667,745.096667,0,0,4.129456,114.554472,10.334362,2.216837,11.496076


## Full oracle-mask KLT run

This run uses the MATLAB block-size contract. It writes an NPZ artifact with pre-Kalman fascicle/aponeurosis geometry and tracker diagnostics, then computes the full-sequence gate metrics.

In [8]:
full_result = run_oracle_mask_klt(limit=mat_n, win_size=contract_win_size, label="full_contract")

out_npz = OUT / "opencv_ultratrack_klt_oracle_mask_features_arrays.npz"
np.savez(
    out_npz,
    frame=full_result["frame"],
    fascicle_segments=full_result["fascicle_segments"],
    sup_apo_lines=full_result["sup_apo_lines"],
    deep_apo_lines=full_result["deep_apo_lines"],
    f_points_count=full_result["f_points_count"],
    a_points_count=full_result["a_points_count"],
    f_inlier_count=full_result["f_inlier_count"],
    a_inlier_count=full_result["a_inlier_count"],
    f_affine_ok=full_result["f_affine_ok"],
    a_affine_ok=full_result["a_affine_ok"],
    f_reinitialized=full_result["f_reinitialized"],
    a_reinitialized=full_result["a_reinitialized"],
    win_size=np.asarray(contract_win_size, dtype=np.int32),
    mm_per_px=np.asarray(mm_per_px, dtype=np.float32),
)
print("Saved:", out_npz)
print({
    "elapsed_s": full_result["elapsed_s"],
    "f_affine_rate": float(np.mean(full_result["f_affine_ok"][1:])),
    "a_affine_rate": float(np.mean(full_result["a_affine_ok"][1:])),
    "f_reinit_count": int(np.sum(full_result["f_reinitialized"])),
    "a_reinit_count": int(np.sum(full_result["a_reinitialized"])),
    "mean_f_points": float(np.mean(full_result["f_points_count"])),
    "mean_a_points": float(np.mean(full_result["a_points_count"])),
})

full_contract: processed 501/2666 frames | fpts=258 apts=669 elapsed=2.6s


full_contract: processed 1001/2666 frames | fpts=256 apts=609 elapsed=4.8s


full_contract: processed 1501/2666 frames | fpts=254 apts=556 elapsed=6.8s


full_contract: processed 2001/2666 frames | fpts=254 apts=548 elapsed=8.7s


full_contract: processed 2501/2666 frames | fpts=254 apts=536 elapsed=10.5s


full_contract: processed 2666/2666 frames | fpts=246 apts=536 elapsed=11.1s
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/opencv_ultratrack_klt_oracle_mask_features_arrays.npz
{'elapsed_s': 11.137174844741821, 'f_affine_rate': 1.0, 'a_affine_rate': 1.0, 'f_reinit_count': 0, 'a_reinit_count': 0, 'mean_f_points': 257.1567891972993, 'mean_a_points': 607.4024756189048}


In [9]:
full_metrics = save_metrics(
    klt_metric_rows(full_result, prefix="oracle_klt"),
    OUT / "klt_oracle_mask_full_metrics.csv",
)
print("Saved:", OUT / "klt_oracle_mask_full_metrics.csv")
display(full_metrics)
print("Full oracle-mask KLT gate passes:", bool(full_metrics["passes"].all()))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/klt_oracle_mask_full_metrics.csv


,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,oracle_klt_x_sup_px,px,2666,8.030072,13.455412,15.809669,0.089799,2.000000,False
1,oracle_klt_y_sup_px,px,2666,6.644637,6.690696,7.196265,0.546170,2.000000,False
2,oracle_klt_x_deep_px,px,2666,-95.911773,96.514322,126.979571,0.656125,2.000000,False
3,oracle_klt_y_deep_px,px,2666,-9.806522,10.117468,11.495614,-0.095481,2.000000,False
4,oracle_klt_angle_deg,deg,2666,-5.070816,5.071523,6.118181,0.709512,1.000000,False
5,oracle_klt_length_px,px,2666,88.415009,94.138829,121.015689,0.643165,22.169625,False
6,oracle_klt_length_mm,mm,2666,7.976229,8.492595,10.917252,0.643165,2.000000,False
7,oracle_klt_super_y1_px,px,2666,-5.911741,5.977666,6.659612,-0.011940,2.000000,False
8,oracle_klt_super_y2_px,px,2666,5.795975,5.799864,6.402313,-0.208715,2.000000,False
9,oracle_klt_deep_y1_px,px,2666,-1.245338,4.575478,5.340453,0.330603,2.000000,False


Full oracle-mask KLT gate passes: False


In [10]:
validation_frames = np.asarray([0, 122, 533, 691, 955, 1066, 1600, 2133, 2665], dtype=int)
validation_result = dict(full_result)
for key in ["fascicle_segments", "sup_apo_lines", "deep_apo_lines"]:
    validation_result[key] = np.asarray(full_result[key])[validation_frames]

mat_klt_validation_backup = mat_klt_segments.copy()
mat_sup_validation_backup = mat_sup_segments.copy()
mat_deep_validation_backup = mat_deep_segments.copy()

# klt_metric_rows compares from the start of the MATLAB arrays, so use an explicit validation metric table here.
est_fas = np.asarray(full_result["fascicle_segments"], dtype=float)[validation_frames]
est_sup = np.asarray(full_result["sup_apo_lines"], dtype=float)[validation_frames]
est_deep = np.asarray(full_result["deep_apo_lines"], dtype=float)[validation_frames]
ref_fas = mat_klt_segments[validation_frames]
ref_sup = mat_sup_segments[validation_frames]
ref_deep = mat_deep_segments[validation_frames]
length_target_px = 2.0 / mm_per_px
validation_rows = [
    metric("oracle_klt_x_sup_px", ref_fas[:, 0], est_fas[:, 0], "px", target_rmse=2.0),
    metric("oracle_klt_y_sup_px", ref_fas[:, 1], est_fas[:, 1], "px", target_rmse=2.0),
    metric("oracle_klt_x_deep_px", ref_fas[:, 2], est_fas[:, 2], "px", target_rmse=2.0),
    metric("oracle_klt_y_deep_px", ref_fas[:, 3], est_fas[:, 3], "px", target_rmse=2.0),
    metric("oracle_klt_angle_deg", normalized_angles_deg(ref_fas), normalized_angles_deg(est_fas), "deg", target_rmse=1.0),
    metric("oracle_klt_length_px", line_lengths_batch(ref_fas), line_lengths_batch(est_fas), "px", target_rmse=length_target_px),
    metric("oracle_klt_length_mm", line_lengths_batch(ref_fas) * mm_per_px, line_lengths_batch(est_fas) * mm_per_px, "mm", target_rmse=2.0),
    metric("oracle_klt_super_y1_px", ref_sup[:, 1], est_sup[:, 1], "px", target_rmse=2.0),
    metric("oracle_klt_super_y2_px", ref_sup[:, 3], est_sup[:, 3], "px", target_rmse=2.0),
    metric("oracle_klt_deep_y1_px", ref_deep[:, 1], est_deep[:, 1], "px", target_rmse=2.0),
    metric("oracle_klt_deep_y2_px", ref_deep[:, 3], est_deep[:, 3], "px", target_rmse=2.0),
]
validation_metrics = save_metrics(validation_rows, OUT / "klt_oracle_mask_validation9_metrics.csv")
print("Saved:", OUT / "klt_oracle_mask_validation9_metrics.csv")
display(validation_metrics)
print("9-frame oracle-mask KLT gate passes:", bool(validation_metrics["passes"].all()))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/klt_oracle_mask_validation9_metrics.csv


,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,oracle_klt_x_sup_px,px,9,3.170725,13.318368,16.071834,-0.619846,2.000000,False
1,oracle_klt_y_sup_px,px,9,4.092164,4.453538,5.010501,0.419532,2.000000,False
2,oracle_klt_x_deep_px,px,9,-24.930656,25.732239,38.167845,0.997784,2.000000,False
3,oracle_klt_y_deep_px,px,9,-9.276039,9.820214,11.111170,-0.437747,2.000000,False
4,oracle_klt_angle_deg,deg,9,-2.288083,2.288083,2.913624,0.996136,1.000000,False
5,oracle_klt_length_px,px,9,20.074205,33.290033,45.129857,0.987792,22.169625,False
6,oracle_klt_length_mm,mm,9,1.810965,3.003211,4.071323,0.987792,2.000000,False
7,oracle_klt_super_y1_px,px,9,-4.676406,4.676406,5.483738,-0.029092,2.000000,False
8,oracle_klt_super_y2_px,px,9,4.239672,4.239672,4.811371,-0.127822,2.000000,False
9,oracle_klt_deep_y1_px,px,9,-3.767069,5.223057,5.976705,0.348573,2.000000,False


9-frame oracle-mask KLT gate passes: False


## Compare against notebook 43 baseline

Notebook 43 compared the older Python KLT artifact against MATLAB. This notebook should reduce that error if the MATLAB masks and KLT settings are closer, even if it does not yet pass the final parity gate.

In [11]:
nb43_klt = pd.read_csv(NB43_KLT_METRICS)
old_angle = float(nb43_klt.loc[nb43_klt["comparison"] == "klt_angle_deg", "rmse"].iloc[0])
old_length_px = float(nb43_klt.loc[nb43_klt["comparison"] == "klt_length_px", "rmse"].iloc[0])
old_length_mm = float(nb43_klt.loc[nb43_klt["comparison"] == "klt_length_mm", "rmse"].iloc[0])
new_angle = float(full_metrics.loc[full_metrics["comparison"] == "oracle_klt_angle_deg", "rmse"].iloc[0])
new_length_px = float(full_metrics.loc[full_metrics["comparison"] == "oracle_klt_length_px", "rmse"].iloc[0])
new_length_mm = float(full_metrics.loc[full_metrics["comparison"] == "oracle_klt_length_mm", "rmse"].iloc[0])

improvement = pd.DataFrame([
    {
        "signal": "fascicle_angle_deg",
        "notebook43_rmse": old_angle,
        "notebook44_rmse": new_angle,
        "absolute_improvement": old_angle - new_angle,
        "relative_improvement_pct": 100.0 * (old_angle - new_angle) / old_angle,
    },
    {
        "signal": "fascicle_length_px",
        "notebook43_rmse": old_length_px,
        "notebook44_rmse": new_length_px,
        "absolute_improvement": old_length_px - new_length_px,
        "relative_improvement_pct": 100.0 * (old_length_px - new_length_px) / old_length_px,
    },
    {
        "signal": "fascicle_length_mm",
        "notebook43_rmse": old_length_mm,
        "notebook44_rmse": new_length_mm,
        "absolute_improvement": old_length_mm - new_length_mm,
        "relative_improvement_pct": 100.0 * (old_length_mm - new_length_mm) / old_length_mm,
    },
])
improvement_path = OUT / "notebook43_to_44_klt_improvement.csv"
improvement.to_csv(improvement_path, index=False)
print("Saved:", improvement_path)
display(improvement)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/notebook43_to_44_klt_improvement.csv


,signal,notebook43_rmse,notebook44_rmse,absolute_improvement,relative_improvement_pct
0,fascicle_angle_deg,10.823260,6.118181,4.705078,43.471915
1,fascicle_length_px,186.941159,121.015689,65.925471,35.265359
2,fascicle_length_mm,16.864621,10.917252,5.947369,35.265359


## Readiness decision

This notebook completes the first KLT porting pass but does not close the KLT gate. The remaining gap is now narrower and more specific:

- OpenCV feature detection is only an approximation of MATLAB `detectMinEigenFeatures` and its metric ordering.
- The LK tracker is not exactly MathWorks `vision.PointTracker`.
- `poly2mask` and OpenCV polygon filling are close but not mathematically identical.
- Full-affine RANSAC behavior may differ from `estimateGeometricTransform2D`.

Next notebook should move this prototype into package code and add lower-level checkpoints: feature counts/locations, affine matrices, and one-step propagation errors against MATLAB for selected frames.

In [12]:
summary = {
    "gate": "UltraTrack/KLT oracle-mask prototype",
    "passes_full_gate": bool(full_metrics["passes"].all()),
    "passes_validation9_gate": bool(validation_metrics["passes"].all()),
    "contract_win_size_width_height": list(contract_win_size),
    "full_frames": int(len(full_result["frame"])),
    "f_affine_rate": float(np.mean(full_result["f_affine_ok"][1:])),
    "a_affine_rate": float(np.mean(full_result["a_affine_ok"][1:])),
    "f_reinit_count": int(np.sum(full_result["f_reinitialized"])),
    "a_reinit_count": int(np.sum(full_result["a_reinitialized"])),
    "notebook43_angle_rmse_deg": old_angle,
    "notebook44_angle_rmse_deg": new_angle,
    "notebook43_length_rmse_px": old_length_px,
    "notebook44_length_rmse_px": new_length_px,
    "notebook43_length_rmse_mm": old_length_mm,
    "notebook44_length_rmse_mm": new_length_mm,
    "next_action": "Package the oracle-mask KLT prototype, then add feature-location and affine one-step checkpoints against MATLAB.",
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

readiness = pd.DataFrame([
    {
        "gate": "TimTrack mask/doHough",
        "status": "passed in notebook 42/43",
        "use_for_next": True,
    },
    {
        "gate": "UltraTrack/KLT oracle-mask prototype",
        "status": "improved but not passed",
        "use_for_next": True,
    },
    {
        "gate": "MATLAB 2-state Kalman",
        "status": "not started; keep separate from current 4-state filter",
        "use_for_next": False,
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
print("Saved:", readiness_path)
display(readiness)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/summary.json
{
  "gate": "UltraTrack/KLT oracle-mask prototype",
  "passes_full_gate": false,
  "passes_validation9_gate": false,
  "contract_win_size_width_height": [
    71,
    21
  ],
  "full_frames": 2666,
  "f_affine_rate": 1.0,
  "a_affine_rate": 1.0,
  "f_reinit_count": 0,
  "a_reinit_count": 0,
  "notebook43_angle_rmse_deg": 10.823259709648635,
  "notebook44_angle_rmse_deg": 6.118181484715813,
  "notebook43_length_rmse_px": 186.9411591466729,
  "notebook44_length_rmse_px": 121.01568860982493,
  "notebook43_length_rmse_mm": 16.864620584940067,
  "notebook44_length_rmse_mm": 10.91725162369773,
  "next_action": "Package the oracle-mask KLT prototype, then add feature-location and affine one-step checkpoints against MATLAB."
}
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/readiness.csv


,gate,status,use_for_next
0,TimTrack mask/doHough,passed in notebook 42/43,True
1,UltraTrack/KLT oracle-mask prototype,improved but not passed,True
2,MATLAB 2-state Kalman,not started; keep separate from current 4-stat...,False
